# Student Performance Prediction using Machine Learning

This notebook walks through a complete beginner-friendly machine learning project using Pandas, NumPy, Matplotlib, and Scikit-learn.

In [ ]:
from pathlib import Path
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = BASE_DIR / 'data' / 'students.csv'
MODEL_PATH = BASE_DIR / 'models' / 'linear_regression.pkl'
PREDICTIONS_PATH = BASE_DIR / 'outputs' / 'predictions.csv'
GRAPH_DIR = BASE_DIR / 'outputs' / 'graphs'

## 1. Load Dataset

We load the dataset using Pandas and inspect the first few rows, shape, data types, and summary statistics.

In [ ]:
data = pd.read_csv(DATA_PATH)
display(data.head())
print('Shape:', data.shape)
data.info()
display(data.describe())

## 2. Data Cleaning

We handle missing values, remove duplicate rows, and correct data types.

In [ ]:
cleaned_data = data.copy()

numeric_columns = [
    'Hours_Studied', 'Attendance', 'Sleep_Hours', 'Previous_Score',
    'Assignments_Completed', 'Exam_Score'
]
categorical_columns = ['Internet_Access', 'Family_Income']

for column in numeric_columns:
    cleaned_data[column] = pd.to_numeric(cleaned_data[column], errors='coerce')
    cleaned_data[column] = cleaned_data[column].fillna(cleaned_data[column].median())

for column in categorical_columns:
    cleaned_data[column] = cleaned_data[column].fillna(cleaned_data[column].mode()[0])

cleaned_data = cleaned_data.drop_duplicates()
cleaned_data['Assignments_Completed'] = cleaned_data['Assignments_Completed'].astype(int)

print(cleaned_data.isnull().sum())
print('Cleaned shape:', cleaned_data.shape)

## 3. Exploratory Data Analysis

The following graphs help us understand the dataset before training the model.

In [ ]:
GRAPH_DIR.mkdir(parents=True, exist_ok=True)

plt.figure(figsize=(8, 5))
plt.hist(cleaned_data['Exam_Score'], bins=20, color='#3A86FF', edgecolor='black')
plt.title('Distribution of Exam Scores')
plt.xlabel('Exam Score')
plt.ylabel('Number of Students')
plt.tight_layout()
plt.savefig(GRAPH_DIR / 'notebook_histogram_exam_scores.png', dpi=150)
plt.show()

print('Explanation: This histogram shows how exam scores are distributed among students.')

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(cleaned_data['Hours_Studied'], cleaned_data['Exam_Score'], alpha=0.6, color='#2A9D8F')
plt.title('Hours Studied vs Exam Score')
plt.xlabel('Hours Studied')
plt.ylabel('Exam Score')
plt.tight_layout()
plt.savefig(GRAPH_DIR / 'notebook_scatter_hours_vs_score.png', dpi=150)
plt.show()

print('Explanation: This scatter plot shows whether students who study more hours usually score higher.')

In [ ]:
plt.figure(figsize=(7, 5))
plt.boxplot(cleaned_data['Exam_Score'], orientation='vertical', patch_artist=True)
plt.title('Box Plot of Exam Scores')
plt.ylabel('Exam Score')
plt.tight_layout()
plt.savefig(GRAPH_DIR / 'notebook_boxplot_exam_scores.png', dpi=150)
plt.show()

print('Explanation: This box plot shows the median, spread, and possible outliers in exam scores.')

In [ ]:
numeric_data = cleaned_data.select_dtypes(include=[np.number])
correlation = numeric_data.corr()

plt.figure(figsize=(9, 7))
plt.imshow(correlation, cmap='coolwarm', interpolation='nearest')
plt.colorbar(label='Correlation')
plt.xticks(range(len(correlation.columns)), correlation.columns, rotation=45)
plt.yticks(range(len(correlation.columns)), correlation.columns)

for row in range(len(correlation.columns)):
    for column in range(len(correlation.columns)):
        plt.text(column, row, f'{correlation.iloc[row, column]:.2f}', ha='center', va='center', fontsize=8)

plt.title('Correlation Heatmap')
plt.tight_layout()
plt.savefig(GRAPH_DIR / 'notebook_correlation_heatmap.png', dpi=150)
plt.show()

print('Explanation: This heatmap shows relationships between numeric columns and Exam_Score.')

In [ ]:
average_scores = cleaned_data.groupby('Internet_Access')['Exam_Score'].mean()

plt.figure(figsize=(7, 5))
plt.bar(average_scores.index, average_scores.values, color=['#E76F51', '#06D6A0'])
plt.title('Average Exam Score by Internet Access')
plt.xlabel('Internet Access')
plt.ylabel('Average Exam Score')
plt.tight_layout()
plt.savefig(GRAPH_DIR / 'notebook_bar_internet_access_scores.png', dpi=150)
plt.show()

print('Explanation: This bar graph compares average exam scores by internet access.')

## 4-7. Feature Selection, Train-Test Split, Model Training, and Evaluation

In [ ]:
feature_columns = [
    'Hours_Studied', 'Attendance', 'Sleep_Hours', 'Previous_Score',
    'Assignments_Completed', 'Internet_Access', 'Family_Income'
]

X = cleaned_data[feature_columns]
y = cleaned_data['Exam_Score']
X_encoded = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=RANDOM_STATE
)

model = LinearRegression()
model.fit(X_train, y_train)
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
mse = mean_squared_error(y_test, predictions)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, predictions)

print(f'MAE: {mae:.2f} - Average absolute prediction error.')
print(f'MSE: {mse:.2f} - Average squared prediction error.')
print(f'RMSE: {rmse:.2f} - Error in the same unit as Exam_Score.')
print(f'R2 Score: {r2:.2f} - Percentage of score variation explained.')

## 8-9. Predictions and Save Model

In [ ]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(MODEL_PATH, 'wb') as file:
    pickle.dump({'model': model, 'feature_columns': list(X_encoded.columns)}, file)

results = X_test.copy()
results['Actual_Exam_Score'] = y_test.values
results['Predicted_Exam_Score'] = predictions.round(2)
results.to_csv(PREDICTIONS_PATH, index=False)

new_student = pd.DataFrame([
    {
        'Hours_Studied': 7.5,
        'Attendance': 88,
        'Sleep_Hours': 7,
        'Previous_Score': 76,
        'Assignments_Completed': 9,
        'Internet_Access': 'Yes',
        'Family_Income': 'Medium',
    }
])
new_student_encoded = pd.get_dummies(new_student, drop_first=True)
new_student_encoded = new_student_encoded.reindex(columns=X_encoded.columns, fill_value=0)

print('Predicted exam score:', round(model.predict(new_student_encoded)[0], 2))